# Bot Behavior Analysis (Round 4)

针对 round 4 trades 中披露的 bot (`Mark XX`) 编号，对每个 bot 在每个 symbol 上的成交进行可视化与 PnL 分解。

图表（共享横轴 timestamp）：
1. **市场图**：wall-mid 曲线 + 该 bot 的买/卖成交点（颜色区分主动/被动方向，大小=数量）。
2. **仓位曲线**：bot 在该 symbol 上的累计持仓。
3. **PnL 曲线 + 分解**：总 PnL（mark-to-market），分解为：
   - `trade_pnl`：成交瞬间相对 wall-mid 的 edge 累计（execution edge）
   - `holding_pnl`：持仓×wall-mid 变动的累计（mark-to-market drift）

**Wall-mid**：每个 timestamp 下，挑选 bid/ask 各自最大订单量的价格相加除二；若单边缺失则前向填充。


In [107]:
import polars as pl
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

DATA_DIR = Path('/Users/leoliu/imc/prosperity/data/bt')
ROUND = 4
DAY = 1   # 可改为 1 / 2 / 3

prices_path = DATA_DIR / f'prices_round_{ROUND}_day_{DAY}.csv'
trades_path = DATA_DIR / f'trades_round_{ROUND}_day_{DAY}.csv'

prices = pl.read_csv(prices_path, separator=';', infer_schema_length=10000)
# 部分档位列在数据全空时会被推断为 String — 强制 cast 为 Int64
_lvl_cols = [f'{s}_{kind}_{i}' for s in ('bid','ask') for kind in ('price','volume') for i in (1,2,3)]
prices = prices.with_columns([pl.col(c).cast(pl.Int64, strict=False) for c in _lvl_cols if c in prices.columns])
trades = pl.read_csv(trades_path, separator=';')
print('prices:', prices.shape, '| trades:', trades.shape)
print('symbols:', sorted(trades['symbol'].unique().to_list()))
print('bots (buyer ∪ seller):', sorted(set(trades['buyer'].unique().to_list()) | set(trades['seller'].unique().to_list())))


prices: (120000, 17) | trades: (1407, 7)
symbols: ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT', 'VEV_4000', 'VEV_4500', 'VEV_5000', 'VEV_5100', 'VEV_5200', 'VEV_5300', 'VEV_5400', 'VEV_5500', 'VEV_6000', 'VEV_6500']
bots (buyer ∪ seller): ['Mark 01', 'Mark 14', 'Mark 22', 'Mark 38', 'Mark 49', 'Mark 55', 'Mark 67']


In [108]:
# ---- compute wall-mid per (symbol, timestamp) ----
# wall-mid: 取 bid/ask 三档中订单量最大的那一档价格相加 / 2
# 若单边全部缺失则前向填充

def compute_wall_mid(df: pl.DataFrame) -> pl.DataFrame:
    bid_p = [pl.col(f'bid_price_{i}') for i in (1, 2, 3)]
    bid_v = [pl.col(f'bid_volume_{i}').fill_null(0) for i in (1, 2, 3)]
    ask_p = [pl.col(f'ask_price_{i}') for i in (1, 2, 3)]
    ask_v = [pl.col(f'ask_volume_{i}').fill_null(0) for i in (1, 2, 3)]

    # argmax across three levels for each side
    def best_price(prices, vols):
        # 返回对应最大 vol 的 price；若所有 vol=0 则 null
        # 用 when/then 链
        v1, v2, v3 = vols
        p1, p2, p3 = prices
        return (
            pl.when((v1 >= v2) & (v1 >= v3) & (v1 > 0)).then(p1)
            .when((v2 >= v3) & (v2 > 0)).then(p2)
            .when(v3 > 0).then(p3)
            .otherwise(None)
        )

    out = df.with_columns([
        best_price(bid_p, bid_v).alias('wall_bid'),
        best_price(ask_p, ask_v).alias('wall_ask'),
    ])
    out = out.sort(['product', 'timestamp']).with_columns([
        pl.col('wall_bid').forward_fill().over('product'),
        pl.col('wall_ask').forward_fill().over('product'),
    ])
    out = out.with_columns(((pl.col('wall_bid') + pl.col('wall_ask')) / 2).alias('wall_mid'))
    return out.select(['timestamp', 'product', 'wall_bid', 'wall_ask', 'wall_mid', 'mid_price'])

wm = compute_wall_mid(prices)
wm.head(5)


timestamp,product,wall_bid,wall_ask,wall_mid,mid_price
i64,str,i64,i64,f64,f64
0,"""HYDROGEL_PACK""",9947,9968,9957.5,9958.0
100,"""HYDROGEL_PACK""",9950,9971,9960.5,9961.0
200,"""HYDROGEL_PACK""",9951,9972,9961.5,9961.0
300,"""HYDROGEL_PACK""",9950,9971,9960.5,9960.0
400,"""HYDROGEL_PACK""",9951,9972,9961.5,9961.0


In [109]:
# ---- per-bot trade extraction ----
# 把一条 trade 从 bot 视角拆成 signed (买入=+, 卖出=-)

def bot_trades(trades: pl.DataFrame, bot: str) -> pl.DataFrame:
    buys = trades.filter(pl.col('buyer') == bot).with_columns([
        pl.lit(1).alias('side'),  # +1 = buy
        pl.col('quantity').cast(pl.Float64).alias('signed_qty'),
    ])
    sells = trades.filter(pl.col('seller') == bot).with_columns([
        pl.lit(-1).alias('side'),
        (-pl.col('quantity').cast(pl.Float64)).alias('signed_qty'),
    ])
    out = pl.concat([buys, sells]).sort(['symbol', 'timestamp'])
    return out.select(['timestamp', 'symbol', 'price', 'quantity', 'side', 'signed_qty', 'buyer', 'seller'])

ALL_BOTS = sorted(set(trades['buyer'].unique().to_list()) | set(trades['seller'].unique().to_list()))
print('bots:', ALL_BOTS)


bots: ['Mark 01', 'Mark 14', 'Mark 22', 'Mark 38', 'Mark 49', 'Mark 55', 'Mark 67']


In [110]:
# ---- PnL decomposition ----
# 给定 bot 在某 symbol 的成交 + wall_mid 时间序列，得到:
#   position(t)      = cumsum(signed_qty)
#   cash(t)          = cumsum(-signed_qty * exec_price)
#   total_pnl(t)     = cash(t) + position(t) * wall_mid(t)
#
# 分解 (在每个 timestamp 步上):
#   trade_pnl_step   = signed_qty * (wall_mid - exec_price)  (execution edge)
#   holding_pnl_step = position_prev * (wall_mid_t - wall_mid_{t-1})  (mark drift on existing book)
#   trade_pnl + holding_pnl ≡ total_pnl  (恒等成立)

def build_pnl_series(bot_sym_trades: pl.DataFrame, wm_sym: pl.DataFrame) -> pl.DataFrame:
    # wm_sym: timestamp, wall_mid (单一 symbol)
    # bot_sym_trades: timestamp, signed_qty, price (单一 symbol, 单一 bot)
    # 在每个 wall_mid 时间戳上聚合 bot 当 ts 的总成交
    if bot_sym_trades.height == 0:
        return pl.DataFrame()
    agg = bot_sym_trades.group_by('timestamp').agg([
        pl.col('signed_qty').sum().alias('signed_qty'),
        # 加权平均成交价
        ((pl.col('price') * pl.col('quantity')).sum() / pl.col('quantity').sum()).alias('avg_price'),
        pl.col('quantity').sum().alias('volume'),
    ])
    df = wm_sym.join(agg, on='timestamp', how='left').sort('timestamp')
    df = df.with_columns([
        pl.col('signed_qty').fill_null(0.0),
        pl.col('avg_price').fill_null(0.0),
        pl.col('volume').fill_null(0.0),
    ])
    df = df.with_columns([
        pl.col('signed_qty').cum_sum().alias('position'),
        (-pl.col('signed_qty') * pl.col('avg_price')).cum_sum().alias('cash'),
    ])
    df = df.with_columns([
        (pl.col('cash') + pl.col('position') * pl.col('wall_mid')).alias('total_pnl'),
        # trade pnl step: signed_qty * (wall_mid - exec_price);  当 signed_qty=0 时为 0
        pl.when(pl.col('signed_qty') != 0)
            .then(pl.col('signed_qty') * (pl.col('wall_mid') - pl.col('avg_price')))
            .otherwise(0.0)
            .alias('trade_pnl_step'),
    ])
    df = df.with_columns([
        pl.col('trade_pnl_step').cum_sum().alias('trade_pnl'),
        (pl.col('total_pnl') - pl.col('trade_pnl_step').cum_sum()).alias('holding_pnl'),
    ])
    return df


In [111]:
# ---- 主绘图函数 ----

def plot_bot_symbol(bot: str, symbol: str, trades: pl.DataFrame, wm: pl.DataFrame):
    bt = bot_trades(trades, bot).filter(pl.col('symbol') == symbol)
    wm_sym = wm.filter(pl.col('product') == symbol).select(['timestamp', 'wall_mid'])
    if bt.height == 0:
        return None
    pnl = build_pnl_series(bt, wm_sym)

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        row_heights=[0.5, 0.2, 0.3],
        subplot_titles=(
            f'{bot} | {symbol} — Market & Trades',
            'Position', 'PnL decomposition (total = trade + holding)'
        ),
    )
    # row1: wall mid + trades
    fig.add_trace(go.Scattergl(
        x=pnl['timestamp'], y=pnl['wall_mid'], mode='lines',
        name='wall_mid', line=dict(color='#888', width=1)
    ), row=1, col=1)
    buys = bt.filter(pl.col('side') == 1)
    sells = bt.filter(pl.col('side') == -1)
    if buys.height:
        fig.add_trace(go.Scattergl(
            x=buys['timestamp'], y=buys['price'], mode='markers',
            name='bot BUY', marker=dict(color='red', symbol='triangle-up',
                                       size=np.clip(buys['quantity'].to_numpy()*1.5, 5, 25)),
            text=[f'qty={q}, vs {s}' for q, s in zip(buys['quantity'], buys['seller'])],
            hovertemplate='ts=%{x}<br>px=%{y}<br>%{text}<extra></extra>'
        ), row=1, col=1)
    if sells.height:
        fig.add_trace(go.Scattergl(
            x=sells['timestamp'], y=sells['price'], mode='markers',
            name='bot SELL', marker=dict(color='green', symbol='triangle-down',
                                        size=np.clip(sells['quantity'].to_numpy()*1.5, 5, 25)),
            text=[f'qty={q}, vs {b}' for q, b in zip(sells['quantity'], sells['buyer'])],
            hovertemplate='ts=%{x}<br>px=%{y}<br>%{text}<extra></extra>'
        ), row=1, col=1)
    # row2: position
    fig.add_trace(go.Scattergl(
        x=pnl['timestamp'], y=pnl['position'], mode='lines',
        name='position', line=dict(color='#1f77b4', width=1.2),
        fill='tozeroy', fillcolor='rgba(31,119,180,0.15)'
    ), row=2, col=1)
    fig.add_hline(y=0, line=dict(color='#aaa', width=0.5, dash='dot'), row=2, col=1)
    # row3: pnl
    fig.add_trace(go.Scattergl(
        x=pnl['timestamp'], y=pnl['total_pnl'], mode='lines',
        name='total_pnl', line=dict(color='black', width=1.5)
    ), row=3, col=1)
    fig.add_trace(go.Scattergl(
        x=pnl['timestamp'], y=pnl['trade_pnl'], mode='lines',
        name='trade_pnl', line=dict(color='#d62728', width=1)
    ), row=3, col=1)
    fig.add_trace(go.Scattergl(
        x=pnl['timestamp'], y=pnl['holding_pnl'], mode='lines',
        name='holding_pnl', line=dict(color='#2ca02c', width=1)
    ), row=3, col=1)
    fig.add_hline(y=0, line=dict(color='#aaa', width=0.5, dash='dot'), row=3, col=1)

    fig.update_layout(
        height=820, hovermode='x unified',
        legend=dict(orientation='h', y=1.04, x=1, xanchor='right'),
        margin=dict(l=40, r=20, t=60, b=40),
    )
    fig.update_xaxes(title_text='timestamp', row=3, col=1)
    fig.update_yaxes(title_text='price', row=1, col=1)
    fig.update_yaxes(title_text='pos', row=2, col=1)
    fig.update_yaxes(title_text='pnl', row=3, col=1)
    return fig


In [112]:
# ---- bot 概览：每个 bot 在哪些 symbol 上交易了多少 ----
summary = (
    pl.concat([
        trades.select(['timestamp', 'symbol', 'quantity', pl.col('buyer').alias('bot')]).with_columns(pl.lit('buy').alias('side')),
        trades.select(['timestamp', 'symbol', 'quantity', pl.col('seller').alias('bot')]).with_columns(pl.lit('sell').alias('side')),
    ])
    .group_by(['bot', 'symbol'])
    .agg([
        pl.len().alias('n_trades'),
        pl.col('quantity').sum().alias('total_qty'),
    ])
    .sort(['bot', 'total_qty'], descending=[False, True])
)
summary


bot,symbol,n_trades,total_qty
str,str,u32,i64
"""Mark 01""","""VELVETFRUIT_EXTRACT""",157,884
"""Mark 01""","""VEV_6500""",98,345
"""Mark 01""","""VEV_6000""",98,345
"""Mark 01""","""VEV_5500""",89,310
"""Mark 01""","""VEV_5400""",76,267
…,…,…,…
"""Mark 38""","""VEV_5100""",1,1
"""Mark 38""","""VEV_4500""",1,1
"""Mark 49""","""VELVETFRUIT_EXTRACT""",40,380


In [113]:
# # ---- 选定 bot + symbol 出图 ----
BOT = 'Mark 55'
SYMBOL = 'VELVETFRUIT_EXTRACT'
fig = plot_bot_symbol(BOT, SYMBOL, trades, wm)
fig.show() if fig is not None else print('no trades')


In [114]:
# ---- 批量：某个 bot 跨所有 symbol 出图 ----
# BOT = 'Mark 38'
# for sym in sorted(trades.filter((pl.col('buyer')==BOT) | (pl.col('seller')==BOT))['symbol'].unique().to_list()):
#     fig = plot_bot_symbol(BOT, sym, trades, wm)
#     if fig is not None:
#         fig.show()


## 附加分析

### 1. Bot 行为指纹 — 成交时相对 wall-mid 的 edge 分布

对每个 bot，在每笔成交瞬间计算 `edge = side * (wall_mid - exec_price)`：
- edge>0：bot 拿到了优于 wall-mid 的价格（更像主动报价的 maker / 或激进单打到了对手挂单的好位置）
- edge<0：bot 付出了相对 wall-mid 的劣势价（典型 taker 行为）

这是判别 bot 是 maker / taker / 信息驱动 / 噪声的快速指纹。


In [115]:
# 把每笔 trade 与 wall_mid (asof on timestamp + product) 对齐
wm_join = wm.select(['timestamp', 'product', 'wall_mid']).rename({'product': 'symbol'})
all_signed = pl.concat([
    trades.with_columns([pl.col('buyer').alias('bot'), pl.lit(1).alias('side'), pl.col('quantity').cast(pl.Float64).alias('signed_qty')]),
    trades.with_columns([pl.col('seller').alias('bot'), pl.lit(-1).alias('side'), (-pl.col('quantity').cast(pl.Float64)).alias('signed_qty')]),
]).select(['timestamp', 'symbol', 'price', 'quantity', 'bot', 'side', 'signed_qty'])
joined = all_signed.join(wm_join, on=['timestamp', 'symbol'], how='left')
joined = joined.with_columns(
    (pl.col('side') * (pl.col('wall_mid') - pl.col('price'))).alias('edge')
)
edge_stats = joined.group_by('bot').agg([
    pl.len().alias('n'),
    pl.col('edge').mean().alias('mean_edge'),
    pl.col('edge').median().alias('med_edge'),
    (pl.col('edge') > 0).mean().alias('pct_positive_edge'),
    pl.col('quantity').sum().alias('total_qty'),
    pl.col('quantity').mean().alias('avg_qty'),
]).sort('mean_edge', descending=True)
edge_stats


bot,n,mean_edge,med_edge,pct_positive_edge,total_qty,avg_qty
str,u32,f64,f64,f64,i64,f64
"""Mark 14""",764,6.76767,7.5,0.989529,3012,3.942408
"""Mark 01""",550,1.267273,0.5,1.0,2261,4.110909
"""Mark 67""",58,1.077586,1.0,1.0,519,8.948276
"""Mark 22""",474,-0.560127,-0.5,0.014768,1806,3.810127
"""Mark 49""",40,-0.8375,-1.0,0.05,380,9.5
"""Mark 55""",384,-2.466146,-2.5,0.010417,2109,5.4921875
"""Mark 38""",544,-8.610294,-8.5,0.011029,1823,3.351103


In [116]:
# Edge 分布直方图（按 bot 分面）
import plotly.express as px
fig = px.histogram(
    joined.drop_nulls('edge').to_pandas(),
    x='edge', color='bot', facet_col='bot', facet_col_wrap=3,
    nbins=60, histnorm='probability density',
    title='Bot edge distribution vs wall-mid (per trade, signed by side)',
)
fig.update_yaxes(matches=None)
fig.update_xaxes(matches=None)
fig.update_layout(height=800, showlegend=False)
fig.show()


### 2. Bot 之间的对手矩阵

对每对 (buyer, seller) 统计成交次数 / 总量。可以看出哪些 bot 经常作为彼此对手 — 暗示策略层的耦合关系。


In [117]:
matrix = (
    trades.group_by(['buyer', 'seller']).agg([pl.len().alias('n'), pl.col('quantity').sum().alias('qty')])
)
import pandas as pd
pivot_qty = matrix.to_pandas().pivot(index='buyer', columns='seller', values='qty').fillna(0)
fig = px.imshow(
    pivot_qty, text_auto=True, aspect='auto', color_continuous_scale='Viridis',
    labels=dict(x='seller', y='buyer', color='qty'),
    title='Counterparty volume matrix (rows=buyer, cols=seller)',
)
fig.update_layout(height=500)
fig.show()


### 3. Bot 终局仓位 / 周转率

- `final_pos`：bot 在该 symbol 收盘持仓 — 接近 0 暗示 mean-reverting / round-trip；持续偏向一侧暗示方向性。
- `turnover = total_volume / |final_pos|`（或 / max|pos|）粗略衡量频次。


In [118]:
pos_summary = (
    all_signed.group_by(['bot', 'symbol']).agg([
        pl.col('signed_qty').sum().alias('final_pos'),
        pl.col('quantity').sum().alias('total_vol'),
        pl.len().alias('n_trades'),
    ])
    .with_columns((pl.col('total_vol') / pl.max_horizontal(pl.col('final_pos').abs(), pl.lit(1))).alias('turnover'))
    .sort(['bot', 'total_vol'], descending=[False, True])
)
pos_summary


bot,symbol,final_pos,total_vol,n_trades,turnover
str,str,f64,i64,u32,f64
"""Mark 01""","""VELVETFRUIT_EXTRACT""",-32.0,884,157,27.625
"""Mark 01""","""VEV_6500""",345.0,345,98,1.0
"""Mark 01""","""VEV_6000""",345.0,345,98,1.0
"""Mark 01""","""VEV_5500""",310.0,310,89,1.0
"""Mark 01""","""VEV_5400""",267.0,267,76,1.0
…,…,…,…,…,…
"""Mark 38""","""VEV_5000""",1.0,1,1,1.0
"""Mark 38""","""VEV_5100""",1.0,1,1,1.0
"""Mark 49""","""VELVETFRUIT_EXTRACT""",-304.0,380,40,1.25


## 4. Trader 信号验证 (mean-reversion benchmark vs trader-conditional)

参考 `trader_signal_validation.md`。

对每个 `(trader_id, product, EMA-deviation bucket)` 三元组,比较:

1. **Benchmark**: 该 bucket 内**所有 tick** 的 `mr_direction × forward_return_100` 均值 (无信号、纯均值回归 baseline)
2. **Conditional Aligned**: trader 在此刻成交且方向与 bucket 一致时的 `signed_return` 均值
3. **Conditional Opposed**: trader 反 bucket 方向成交时的 `signed_return` 均值

约束: 价格用 `wall_mid` (无 spread); 等权; 每个 product 独立切分位; 任一子集 < 30 标 `insufficient`。


In [119]:
# ---- 多日合并 setup: 用 day 1+2+3 全量数据跑后续验证 ----
# 关键: EMA / forward_return / wall_mid 都在 (product, day) 内独立计算,
# trades 也带 day 列, join 时 key 加上 day, 杜绝跨日泄漏。
# 前面的 plotting / edge 分析仍使用 day-1 单日 (`wm`, `trades`, `all_signed`)

ALL_DAYS = (1, 2, 3)

def _load_day(d):
    p = pl.read_csv(DATA_DIR / f'prices_round_{ROUND}_day_{d}.csv',
                    separator=';', infer_schema_length=10000)
    p = p.with_columns([pl.col(c).cast(pl.Int64, strict=False)
                        for c in _lvl_cols if c in p.columns])
    t = pl.read_csv(DATA_DIR / f'trades_round_{ROUND}_day_{d}.csv', separator=';')
    return (p.with_columns(pl.lit(d).alias('day')),
            t.with_columns(pl.lit(d).alias('day')))

_p_parts, _t_parts = zip(*(_load_day(d) for d in ALL_DAYS))
prices_multi = pl.concat(_p_parts)
trades_multi = pl.concat(_t_parts)

# 复用 compute_wall_mid: 每天独立算后再 concat 上 day 列
wm_multi = pl.concat([
    compute_wall_mid(prices_multi.filter(pl.col('day') == d).drop('day'))
        .with_columns(pl.lit(d).alias('day'))
    for d in ALL_DAYS
])

# trader 视角带 day 的 signed trades
signed_multi = pl.concat([
    trades_multi.with_columns([
        pl.col('buyer').alias('bot'),
        pl.lit(1).alias('side'),
        pl.col('quantity').cast(pl.Float64).alias('signed_qty'),
    ]),
    trades_multi.with_columns([
        pl.col('seller').alias('bot'),
        pl.lit(-1).alias('side'),
        (-pl.col('quantity').cast(pl.Float64)).alias('signed_qty'),
    ]),
]).select(['day', 'timestamp', 'symbol', 'price', 'quantity', 'bot', 'side', 'signed_qty'])

print(f'wm_multi: {wm_multi.shape} | signed_multi: {signed_multi.shape}')
trades_multi.group_by('day').agg(pl.len().alias('n_trades')).sort('day')


wm_multi: (360000, 7) | signed_multi: (8562, 8)


day,n_trades
i32,u32
1,1407
2,1333
3,1541


In [120]:
# ---- §1.1 + §2: 每个 (product, day, timestamp) 计算 mid / ema / deviation / forward_return ----
# tick 步长 = 100 ts; "100-tick 前瞻" 即 shift(-100) (每行一个 tick, per product/day 已排序)

EMA_PERIOD = 4000
FWD_TICKS = 100
ALPHA = 2.0 / (EMA_PERIOD + 1)

wm_ts = (
    wm_multi.select(['day', 'timestamp', 'product', 'wall_mid'])
      .sort(['product', 'day', 'timestamp'])
      .with_columns(
          pl.col('wall_mid').ewm_mean(alpha=ALPHA, adjust=False).over(['product', 'day']).alias('ema'),
      )
      .with_columns(
          ((pl.col('wall_mid') - pl.col('ema')) / pl.col('ema')).alias('deviation'),
          (pl.col('wall_mid').shift(-FWD_TICKS).over(['product', 'day']) / pl.col('wall_mid') - 1.0)
              .alias('forward_return_100'),
      )
)
print('wm_ts shape:', wm_ts.shape)
wm_ts.head(5)


wm_ts shape: (360000, 7)


day,timestamp,product,wall_mid,ema,deviation,forward_return_100
i32,i64,str,f64,f64,f64,f64
1,0,"""HYDROGEL_PACK""",9957.5,9957.5,0.0,-0.001657
1,100,"""HYDROGEL_PACK""",9960.5,9957.5015,0.000301,-0.001606
1,200,"""HYDROGEL_PACK""",9961.5,9957.503498,0.000401,-0.001606
1,300,"""HYDROGEL_PACK""",9960.5,9957.504996,0.000301,-0.001506
1,400,"""HYDROGEL_PACK""",9961.5,9957.506993,0.000401,-0.001757


In [121]:
# ---- §3.1: 每个 (product, day) 独立切 N_BUCKETS 档 quantile ----
# N_BUCKETS=3 → 底/中/顶 三档,只用首尾,中间剔除 (mr_direction=0)
# N_BUCKETS=5 → 原 5 档方案 (B1..B5),首尾各 20%
# 调小 N_BUCKETS 缓解每格样本不足 (单天 1407 笔 trade,7 bot × 12 symbol 已经稀疏)

N_BUCKETS = 5

QS = [i / N_BUCKETS for i in range(1, N_BUCKETS)]
boundaries = (
    wm_ts.drop_nulls('deviation')
         .group_by(['product', 'day'])
         .agg([pl.col('deviation').quantile(q).alias(f'q{i+1}') for i, q in enumerate(QS)])
)
print(f'per-product/day deviation quantile cuts (N_BUCKETS={N_BUCKETS}, qs={QS}):')
boundaries


per-product/day deviation quantile cuts (N_BUCKETS=5, qs=[0.2, 0.4, 0.6, 0.8]):


product,day,q1,q2,q3,q4
str,i32,f64,f64,f64,f64
"""VEV_5000""",1,-0.037964,-0.00886,0.01405,0.050608
"""HYDROGEL_PACK""",1,-0.001281,0.000005,0.001594,0.004101
"""VEV_6000""",2,0.0,0.0,0.0,0.0
"""VEV_5000""",3,-0.107704,-0.079799,-0.020584,0.041687
"""VEV_5100""",1,-0.05522,-0.015493,0.014598,0.065659
…,…,…,…,…,…
"""VELVETFRUIT_EXTRACT""",1,-0.001886,-0.00037,0.000824,0.002681
"""HYDROGEL_PACK""",2,-0.002073,-0.000434,0.000827,0.002083
"""VEV_5200""",3,-0.21533,-0.162803,-0.057117,0.06822


In [122]:
# ---- §3.1 + §3.2: 给每个 tick 打 bucket / mr_direction 标签 ----
# 首档 (B1) → mr_direction=+1 (买); 末档 (B{N_BUCKETS}) → -1 (卖); 中间档 → 0 (剔除)

_q_cols = [f'q{i+1}' for i in range(len(QS))]

def _bucket_expr_dynamic(dev, q_col_names):
    e = pl.when(dev < pl.col(q_col_names[0])).then(pl.lit('B1'))
    for i, qn in enumerate(q_col_names[1:], start=2):
        e = e.when(dev < pl.col(qn)).then(pl.lit(f'B{i}'))
    return e.otherwise(pl.lit(f'B{len(q_col_names) + 1}'))

_MR_MAP = {'B1': 1, f'B{N_BUCKETS}': -1}
for i in range(2, N_BUCKETS):
    _MR_MAP[f'B{i}'] = 0

wm_bucket = (
    wm_ts.join(boundaries, on=['product', 'day'], how='left')
         .with_columns(_bucket_expr_dynamic(pl.col('deviation'), _q_cols).alias('bucket'))
         .with_columns(
             pl.col('bucket').replace_strict(_MR_MAP, return_dtype=pl.Int8).alias('mr_direction')
         )
         .drop(_q_cols)
)
print('bucket counts per product/day:')
wm_bucket.group_by(['product', 'day', 'bucket']).len().sort(['product', 'day', 'bucket'])


bucket counts per product/day:


product,day,bucket,len
str,i32,str,u32
"""HYDROGEL_PACK""",1,"""B1""",2000
"""HYDROGEL_PACK""",1,"""B2""",2000
"""HYDROGEL_PACK""",1,"""B3""",1999
"""HYDROGEL_PACK""",1,"""B4""",2000
"""HYDROGEL_PACK""",1,"""B5""",2001
…,…,…,…
"""VEV_6000""",2,"""B5""",10000
"""VEV_6000""",3,"""B5""",10000
"""VEV_6500""",1,"""B5""",10000


In [123]:
# ---- §1.2 + §3.2: 把 trader 成交挂上 bucket / alignment / signed_return ----
# signed_multi 已包含 (day, timestamp, symbol, bot, side, signed_qty, price)

trader_tagged = (
    signed_multi
    .join(
        wm_bucket.rename({'product': 'symbol'})
                 .select(['day', 'timestamp', 'symbol', 'wall_mid', 'ema', 'deviation',
                          'bucket', 'mr_direction', 'forward_return_100']),
        on=['day', 'timestamp', 'symbol'], how='left',
    )
    .with_columns(
        pl.when(pl.col('side') == pl.col('mr_direction')).then(pl.lit('aligned'))
        .when(pl.col('side') == -pl.col('mr_direction')).then(pl.lit('opposed'))
        .otherwise(pl.lit('neutral'))
        .alias('alignment'),
        (pl.col('side') * pl.col('forward_return_100')).alias('signed_return'),
    )
)
print('trader_tagged shape:', trader_tagged.shape)
trader_tagged.group_by(['bucket', 'alignment']).len().sort(['bucket', 'alignment'])


trader_tagged shape: (8562, 16)


bucket,alignment,len
str,str,u32
"""B1""","""aligned""",752
"""B1""","""opposed""",752
"""B2""","""neutral""",1522
"""B3""","""neutral""",1410
"""B4""","""neutral""",1458
"""B5""","""aligned""",1334
"""B5""","""opposed""",1334


In [124]:
# ---- §4.1: benchmark per (product, bucket) — 无信号 mean-reversion baseline ----

bench = (
    wm_bucket
    .drop_nulls(['forward_return_100', 'bucket'])
    .with_columns((pl.col('mr_direction') * pl.col('forward_return_100')).alias('mr_return'))
    .group_by(['product', 'bucket'])
    .agg([
        pl.len().alias('n_ticks_in_bucket'),
        pl.col('mr_return').mean().alias('benchmark'),
    ])
    .sort(['product', 'bucket'])
)
bench


product,bucket,n_ticks_in_bucket,benchmark
str,str,u32,f64
"""HYDROGEL_PACK""","""B1""",6000,0.000525
"""HYDROGEL_PACK""","""B2""",5981,0.0
"""HYDROGEL_PACK""","""B3""",5902,0.0
"""HYDROGEL_PACK""","""B4""",5899,0.0
"""HYDROGEL_PACK""","""B5""",5918,0.000553
…,…,…,…
"""VEV_5500""","""B3""",5912,0.0
"""VEV_5500""","""B4""",5900,0.0
"""VEV_5500""","""B5""",5889,0.041874


### 如何看下面这张表

每一行 = 一个 `(trader_id, product, bucket)` 三元组的一次"信号检验"。

**bucket 切分**: 上一格的 `N_BUCKETS` 控制档数。默认 `N_BUCKETS=3`,即按 deviation 33% / 67% 分位切成 **B1**(底,negative deviation,均值回归该买) / **B2**(中,剔除) / **B3**(顶,positive deviation,均值回归该卖)。设 `N_BUCKETS=5` 退回原 5 档方案。**只首尾两档进入下表**,中间档 `mr_direction=0`,无信号方向。

**列含义速查**

| 列 | 含义 |
|---|---|
| `bucket` | 该 product 的 EMA 偏离档位。**B1**=底档(price 远低于 EMA → mr 该买);**B{N}**(末档)=顶档(price 远高于 EMA → mr 该卖) |
| `n_ticks_in_bucket` | 该 product 在此 bucket 一共多少个 tick — benchmark 的样本量 |
| `n_aligned` | 该 trader 在此 bucket 内**与均值回归同向**成交的笔数(底档买 / 顶档卖) |
| `n_opposed` | 该 trader **逆均值回归**成交的笔数(底档卖 / 顶档买) |
| `benchmark` | 不看 trader,该 bucket 内每个 tick 按 mr 方向算的 100-tick 前瞻收益均值。**正数说明均值回归本身能赚钱**,这是要击败的 baseline |
| `conditional_aligned` | 只挑 trader aligned 成交的 tick,按 mr 方向算的前瞻收益均值 |
| `conditional_opposed` | 只挑 trader opposed 成交的 tick,按 **trader 方向**算的前瞻收益均值(>0 意味着 trader 反向时反而对) |
| `aligned_minus_benchmark` | aligned 比 baseline 多赚多少。**>0 且显著 → trader 是确认信号** |
| `aligned_ci_low/high` | conditional_aligned 的 bootstrap 95% 置信区间 |
| `opposed_ci_low/high` | conditional_opposed 的 95% CI |
| `decision` | `confirm`=aligned CI 下界 > benchmark,trader 是有效确认信号;`ignore`=与 baseline 无差;`fade_mr`=opposed CI 下界 > 0,trader 反向时该跟 trader;`insufficient`=两侧都 < 30 笔 |

**怎么读一行**

举例(N_BUCKETS=3): `Mark 55 / VELVETFRUIT_EXTRACT / B1`
- B1 = 价格深度低于 EMA → 均值回归该**买**
- `n_aligned` → Mark 55 在这种行情下**买**了多少次
- `benchmark` → 单纯按"低于 EMA 就买"每 100tick 的平均收益
- `conditional_aligned > benchmark` 且 CI 下界也 > benchmark → Mark 55 一买,平均赚得**显著多于** baseline → `decision=confirm`

**怎么用这张表**
1. 先看 `decision` 列,只关注 `confirm` / `fade_mr`
2. `aligned_minus_benchmark` 排序,挑超额最大的 (trader, product, bucket) 组合
3. 如果 `insufficient` 仍然多,可继续调小 `N_BUCKETS=2`(只按符号分档,无中间档),或合并 day 1/2/3 数据


In [125]:
# ---- §4.2 + §4.3 + §6: conditional means + bootstrap CI + 决策 ----

import pandas as pd

MIN_N = 5
N_BOOT = 1000
CI = 0.90
rng = np.random.default_rng(42)

def boot_mean_ci(arr: np.ndarray, n_boot: int = N_BOOT, ci: float = CI):
    n = len(arr)
    if n < 2:
        return (np.nan, np.nan)
    idx = rng.integers(0, n, size=(n_boot, n))
    means = arr[idx].mean(axis=1)
    lo = float(np.quantile(means, (1 - ci) / 2))
    hi = float(np.quantile(means, 1 - (1 - ci) / 2))
    return (lo, hi)

# 剔除 mr_direction=0 的中性档 (N_BUCKETS=3 时即 B2, =5 时即 B3)
pdf = (
    trader_tagged
    .drop_nulls(['forward_return_100', 'bucket'])
    .filter(pl.col('mr_direction') != 0)
    .select(['bot', 'symbol', 'bucket', 'alignment', 'signed_return'])
    .to_pandas()
)

records = []
for (bot, sym, bucket), g in pdf.groupby(['bot', 'symbol', 'bucket']):
    aligned = g.loc[g.alignment == 'aligned', 'signed_return'].to_numpy()
    opposed = g.loc[g.alignment == 'opposed', 'signed_return'].to_numpy()
    a_lo, a_hi = boot_mean_ci(aligned)
    o_lo, o_hi = boot_mean_ci(opposed)
    records.append({
        'trader_id': bot, 'product': sym, 'bucket': bucket,
        'n_aligned': len(aligned), 'n_opposed': len(opposed),
        'conditional_aligned': aligned.mean() if len(aligned) else np.nan,
        'conditional_opposed': opposed.mean() if len(opposed) else np.nan,
        'aligned_ci_low': a_lo, 'aligned_ci_high': a_hi,
        'opposed_ci_low': o_lo, 'opposed_ci_high': o_hi,
    })

result = pd.DataFrame(records).merge(
    bench.to_pandas(), on=['product', 'bucket'], how='left'
)
result['aligned_minus_benchmark'] = result['conditional_aligned'] - result['benchmark']

def _decide(r):
    parts = []
    if r['n_aligned'] >= MIN_N:
        # CI 下界严格高于 benchmark → confirm
        if r['aligned_ci_low'] > r['benchmark']:
            parts.append('confirm')
        else:
            parts.append('ignore')
    if r['n_opposed'] >= MIN_N:
        # opposed return 显著 > 0 → 跟 trader 反向 = fade_mr
        if r['opposed_ci_low'] > 0:
            parts.append('fade_mr')
    return '+'.join(parts) if parts else 'insufficient'

result['decision'] = result.apply(_decide, axis=1)
result = result[[
    'trader_id', 'product', 'bucket',
    'n_ticks_in_bucket', 'n_aligned', 'n_opposed',
    'benchmark', 'conditional_aligned', 'conditional_opposed',
    'aligned_minus_benchmark',
    'aligned_ci_low', 'aligned_ci_high',
    'opposed_ci_low', 'opposed_ci_high',
    'decision',
]].sort_values(['trader_id', 'product', 'bucket']).reset_index(drop=True)

print('total rows:', len(result),
      '| confirms:', (result.decision.str.contains('confirm')).sum(),
      '| fade_mr:', (result.decision.str.contains('fade_mr')).sum(),
      '| insufficient:', (result.decision == 'insufficient').sum())


with pd.option_context('display.max_rows', None):
    display(result)


total rows: 59 | confirms: 2 | fade_mr: 0 | insufficient: 31


,trader_id,product,bucket,n_ticks_in_bucket,n_aligned,n_opposed,benchmark,conditional_aligned,conditional_opposed,aligned_minus_benchmark,aligned_ci_low,aligned_ci_high,opposed_ci_low,opposed_ci_high,decision
0,Mark 01,VELVETFRUIT_EXTRACT,B1,6000,46,54,0.000562,0.000655,-0.000693,0.000093,0.000248,0.001069,-0.001060,-0.000317,ignore
1,Mark 01,VELVETFRUIT_EXTRACT,B5,5877,42,55,0.000482,0.000769,-0.000946,0.000287,0.000386,0.001156,-0.001347,-0.000537,ignore
2,Mark 01,VEV_5200,B1,6000,6,0,0.024491,0.015582,NaN,-0.008908,-0.028435,0.051489,NaN,NaN,ignore
3,Mark 01,VEV_5200,B5,5878,0,1,0.015243,NaN,-0.032468,NaN,NaN,NaN,NaN,NaN,insufficient
4,Mark 01,VEV_5300,B1,6000,35,0,0.034384,0.023510,NaN,-0.010874,0.001885,0.044568,NaN,NaN,ignore
5,Mark 01,VEV_5300,B5,5879,0,18,0.020310,NaN,0.021266,NaN,NaN,NaN,-0.014739,0.056500,insufficient
6,Mark 01,VEV_5400,B1,6000,64,0,0.043264,0.040410,NaN,-0.002854,0.014503,0.067079,NaN,NaN,ignore
7,Mark 01,VEV_5400,B5,5879,0,39,0.026342,NaN,-0.007983,NaN,NaN,NaN,-0.044270,0.028941,insufficient
8,Mark 01,VEV_5500,B1,6000,66,0,0.132596,0.136370,NaN,0.003774,0.077735,0.205827,NaN,NaN,ignore
9,Mark 01,VEV_5500,B5,5889,0,58,0.041874,NaN,-0.048270,NaN,NaN,NaN,-0.073632,-0.022654,insufficient


In [126]:
# ---- 仅看有效决策行 (samples >= MIN_N) ----
actionable = result[
    (result.decision != 'insufficient') &
    (result.decision != 'ignore')
].sort_values('aligned_minus_benchmark', ascending=False)
actionable


,trader_id,product,bucket,n_ticks_in_bucket,n_aligned,n_opposed,benchmark,conditional_aligned,conditional_opposed,aligned_minus_benchmark,aligned_ci_low,aligned_ci_high,opposed_ci_low,opposed_ci_high,decision
12,Mark 14,HYDROGEL_PACK,B1,6000,87,118,0.000525,0.000952,-0.000580,0.000427,0.000578,0.001308,-0.000856,-0.000321,confirm
56,Mark 55,VELVETFRUIT_EXTRACT,B5,5877,115,115,0.000482,0.000811,-0.000455,0.000329,0.000551,0.001058,-0.000722,-0.000174,confirm
